In [3]:

# YOUTUBE INDIA - SQL ANALYSIS

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install SQL library
!pip install pandasql -q

from pandasql import sqldf
import pandas as pd

pysqldf = lambda q: sqldf(q, globals())

# Load the CLEANED data
youtube_data = pd.read_csv(
    '/content/drive/MyDrive/Youtube Project/youtube_clean_data.csv'
)

print(" Clean Data Loaded!")
print(f"Total Rows: {youtube_data.shape[0]}")
print(f"Columns: {youtube_data.columns.tolist()}")


# Remove None category rows
youtube_data_clean = youtube_data[
    youtube_data['category_name'].notna()
].copy()

print(f"Before: {youtube_data.shape[0]} rows")
print(f"After removing None: {youtube_data_clean.shape[0]} rows")
print(f"Removed: {youtube_data.shape[0] - youtube_data_clean.shape[0]} rows")

youtube_data = youtube_data_clean



# QUERY 1 - CATEGORY PERFORMANCE + RANK

query1 = """
SELECT
    category_name,
    COUNT(*) AS total_videos,
    ROUND(AVG(view_count), 0) AS avg_views,
    ROUND(AVG(engagement_rate), 2) AS avg_engagement,
    RANK() OVER (ORDER BY AVG(view_count) DESC) AS view_rank,
    RANK() OVER (ORDER BY AVG(engagement_rate) DESC) AS engagement_rank
FROM youtube_data
GROUP BY category_name
ORDER BY view_rank
"""
result1 = pysqldf(query1)
print("=== QUERY 1: CATEGORY PERFORMANCE + RANK ===")
print(result1)
print("\n" + "="*60 + "\n")


# QUERY 2 - BEST UPLOAD TIME WINDOW

query2 = """
SELECT
    CASE
        WHEN publish_hour_IST BETWEEN 6 AND 11 THEN 'Morning'
        WHEN publish_hour_IST BETWEEN 12 AND 17 THEN 'Afternoon'
        WHEN publish_hour_IST BETWEEN 18 AND 23 THEN 'Evening'
        ELSE 'Late Night'
    END AS time_window,
    COUNT(*) AS total_videos,
    ROUND(AVG(view_count), 0) AS avg_views,
    ROUND(AVG(engagement_rate), 2) AS avg_engagement
FROM youtube_data
GROUP BY time_window
ORDER BY avg_views DESC
"""
result2 = pysqldf(query2)
print("=== QUERY 2: BEST UPLOAD TIME WINDOW ===")
print(result2)
print("\n" + "="*60 + "\n")



# QUERY 3 - CONSISTENT TOP CHANNELS

query3 = """
SELECT
    channelTitle,
    COUNT(*) AS trending_count,
    ROUND(AVG(view_count), 0) AS avg_views,
    ROUND(MIN(view_count), 0) AS min_views,
    ROUND(MAX(view_count), 0) AS max_views,
    ROUND(MAX(view_count) - MIN(view_count), 0) AS performance_range
FROM youtube_data
GROUP BY channelTitle
HAVING COUNT(*) >= 10
ORDER BY trending_count DESC
LIMIT 10
"""
result3 = pysqldf(query3)
print("=== QUERY 3: CONSISTENT TOP CHANNELS ===")
print(result3)
print("\n" + "="*60 + "\n")



# QUERY 4 - ENGAGEMENT THRESHOLD DETECTION

query4 = """
SELECT
    CASE
        WHEN engagement_rate >= 10 THEN 'Viral'
        WHEN engagement_rate >= 6 THEN 'High'
        WHEN engagement_rate >= 3 THEN 'Average'
        ELSE 'Low'
    END AS engagement_tier,
    COUNT(*) AS total_videos,
    ROUND(AVG(view_count), 0) AS avg_views,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM youtube_data), 2) AS percent_share
FROM youtube_data
GROUP BY engagement_tier
ORDER BY avg_views DESC
"""
result4 = pysqldf(query4)
print("=== QUERY 4: ENGAGEMENT THRESHOLD ===")
print(result4)
print("\n" + "="*60 + "\n")


# QUERY 5 - TITLE LENGTH OPTIMIZATION

query5 = """
SELECT
    CASE
        WHEN title_length < 30 THEN 'Very Short'
        WHEN title_length < 50 THEN 'Short'
        WHEN title_length < 70 THEN 'Medium'
        ELSE 'Long'
    END AS title_bucket,
    COUNT(*) AS total_videos,
    ROUND(AVG(view_count), 0) AS avg_views,
    ROUND(AVG(engagement_rate), 2) AS avg_engagement
FROM youtube_data
GROUP BY title_bucket
ORDER BY avg_views DESC
"""
result5 = pysqldf(query5)
print("=== QUERY 5: TITLE LENGTH OPTIMIZATION ===")
print(result5)
print("\n" + "="*60 + "\n")



# QUERY 6 - YEAR OVER YEAR GROWTH

query6 = """
SELECT
    trend_year,
    avg_views,
    LAG(avg_views) OVER (ORDER BY trend_year) AS prev_year_views,
    ROUND(
        (avg_views - LAG(avg_views) OVER (ORDER BY trend_year))
        / LAG(avg_views) OVER (ORDER BY trend_year) * 100, 2
    ) AS yoy_growth_percent
FROM (
    SELECT
        SUBSTR(trending_date, 1, 4) AS trend_year,
        ROUND(AVG(view_count), 0) AS avg_views
    FROM youtube_data
    GROUP BY trend_year
) yearly_data
ORDER BY trend_year
"""
result6 = pysqldf(query6)
print("=== QUERY 6: YEAR OVER YEAR GROWTH ===")
print(result6)
print("\n" + "="*60 + "\n")



# QUERY 7 - CONTENT STRATEGY SCORECARD
query7 = """
WITH category_agg AS (
    SELECT
        category_name,
        AVG(view_count) AS avg_views,
        AVG(engagement_rate) AS avg_engagement,
        AVG(days_to_trend) AS avg_days_to_trend,
        COUNT(*) AS total_videos
    FROM youtube_data
    GROUP BY category_name
    HAVING COUNT(*) >= 100
),
ranked AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY avg_views DESC) AS views_quartile,
        NTILE(4) OVER (ORDER BY avg_engagement DESC) AS engagement_quartile
    FROM category_agg
),
scored AS (
    SELECT
        *,
        views_quartile + engagement_quartile AS combined_score
    FROM ranked
)
SELECT
    RANK() OVER (ORDER BY combined_score ASC, avg_views DESC) AS category_rank,
    category_name,
    total_videos,
    ROUND(avg_views, 0) AS avg_views,
    ROUND(avg_engagement, 2) AS avg_engagement,
    ROUND(avg_days_to_trend, 2) AS avg_days_to_trend,
    views_quartile,
    engagement_quartile,
    combined_score,
    CASE
        WHEN combined_score <= 3 THEN 1
        WHEN combined_score <= 5 THEN 2
        WHEN combined_score = 6 THEN 3
        ELSE 4
    END AS priority_rank,
    CASE
        WHEN combined_score <= 3 THEN 'High Priority'
        WHEN combined_score <= 5 THEN 'Growth Potential'
        WHEN combined_score = 6 THEN 'Stable'
        ELSE 'Needs Improvement'
    END AS strategy_recommendation
FROM scored
ORDER BY category_rank
"""
result7 = pysqldf(query7)

print("=== QUERY 7: CONTENT STRATEGY SCORECARD ===")
print(result7)
print("\n" + "="*60 + "\n")


# SAVE ALL RESULTS

result1.to_csv('/content/drive/MyDrive/Youtube Project/sql_q1_category_rank.csv', index=False)
result2.to_csv('/content/drive/MyDrive/Youtube Project/sql_q2_upload_time.csv', index=False)
result3.to_csv('/content/drive/MyDrive/Youtube Project/sql_q3_top_channels.csv', index=False)
result4.to_csv('/content/drive/MyDrive/Youtube Project/sql_q4_engagement_tier.csv', index=False)
result5.to_csv('/content/drive/MyDrive/Youtube Project/sql_q5_title_length.csv', index=False)
result6.to_csv('/content/drive/MyDrive/Youtube Project/sql_q6_yearly_growth.csv', index=False)
result7.to_csv('/content/drive/MyDrive/Youtube Project/sql_q7_strategy.csv', index=False)

print(" ALL 7 SQL RESULTS SAVED TO DRIVE!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Clean Data Loaded!
Total Rows: 78847
Columns: ['title', 'channelTitle', 'category_name', 'trending_date', 'publish_hour_IST', 'view_count', 'likes', 'dislikes', 'comment_count', 'engagement_rate', 'title_length', 'days_to_trend', 'tags_count']
Before: 78847 rows
After removing None: 78821 rows
Removed: 26 rows
=== QUERY 1: CATEGORY PERFORMANCE + RANK ===
           category_name  total_videos  avg_views  avg_engagement  view_rank  \
0         Pets & Animals            30  3402380.0            7.02          1   
1                  Music          8012  1956834.0            8.25          2   
2       Film & Animation          1607  1855686.0            5.43          3   
3   Science & Technology          2477  1801976.0           10.02          4   
4                 Sports          2403  1397793.0            4.78          5   
5          Entertainment         